# 人体动作识别 — LSTM (561 维特征)

使用 UCI HAR Dataset 的 561 维手工提取特征，将其视为长度为 561 的单通道序列，通过 LSTM 网络识别 6 种日常动作。

## 1. 导入依赖

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

## 2. 数据加载

直接读取 UCI HAR Dataset 中已提取好的 561 维特征（`X_train.txt` / `X_test.txt`），reshape 为 `(样本数, 561, 1)` 供 LSTM 使用，并从训练集中划分 20% 作为验证集。

In [2]:
def load_data():
    print("正在加载 561维 特征数据...")

    x_train = pd.read_csv('UCI HAR Dataset/train/X_train.txt', header=None, sep='\\s+').values
    y_train = pd.read_csv('UCI HAR Dataset/train/y_train.txt', header=None, sep='\\s+').values

    x_test  = pd.read_csv('UCI HAR Dataset/test/X_test.txt',   header=None, sep='\\s+').values
    y_test  = pd.read_csv('UCI HAR Dataset/test/y_test.txt',   header=None, sep='\\s+').values

    # 从原训练集中划分验证集（20%）
    n_train = x_train.shape[0]
    indices = np.random.permutation(n_train)
    val_size = int(n_train * 0.2)
    val_indices = indices[:val_size]
    train_indices = indices[val_size:]

    x_val = x_train[val_indices]
    y_val = y_train[val_indices]
    x_train = x_train[train_indices]
    y_train = y_train[train_indices]

    # reshape 为 (N, 561, 1) 供 LSTM 使用
    x_train = x_train.reshape(x_train.shape[0], x_train.shape[1], 1)
    x_val   = x_val.reshape(x_val.shape[0], x_val.shape[1], 1)
    x_test  = x_test.reshape(x_test.shape[0], x_test.shape[1], 1)

    # 标签处理: 1-6 转 0-5，再转 One-Hot
    y_train = to_categorical(y_train - 1)
    y_val   = to_categorical(y_val - 1)
    y_test  = to_categorical(y_test - 1)

    print(f"加载完成: 训练集 {x_train.shape}, 验证集 {x_val.shape}, 测试集 {x_test.shape}")
    return x_train, y_train, x_val, y_val, x_test, y_test

In [3]:
x_train, y_train, x_val, y_val, x_test, y_test = load_data()

正在加载 561维 特征数据...
加载完成: 训练集 (5882, 561, 1), 验证集 (1470, 561, 1), 测试集 (2947, 561, 1)


## 3. 模型结构 — LSTM

两层 LSTM 提取时序特征，最后通过全连接层分类：

```text
输入 (561, 1)
  → LSTM(128, return_sequences=True) → Dropout(0.3)
  → LSTM(64) → Dropout(0.3)
  → Dense(64, relu) → Dropout(0.5)
  → Softmax(6)
```

In [4]:
def create_model(input_shape, n_classes):
    inputs = Input(shape=input_shape)
    x = LSTM(128, return_sequences=True,
             kernel_regularizer=l2(1e-4))(inputs)
    x = Dropout(0.3)(x)
    x = LSTM(64,
             kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(n_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(1e-3),
                  metrics=['accuracy'])
    return model

model = create_model(input_shape=x_train.shape[1:], n_classes=y_train.shape[1])
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 561, 1)]          0         
                                                                 
 lstm (LSTM)                 (None, 561, 128)          66560     
                                                                 
 dropout (Dropout)           (None, 561, 128)          0         
                                                                 
 lstm_1 (LSTM)               (None, 64)                49408     
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense (Dense)               (None, 64)                4160      
                                                                 
 batch_normalization (BatchN  (None, 64)               256   

## 4. 训练与评估

In [ ]:
model.fit(
    x_train, y_train,
    epochs=100, batch_size=64, validation_data=(x_val, y_val),
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
    ],
    verbose=1,
)

loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f'测试集准确率: {accuracy * 100:.2f}%')

Epoch 1/100
92/92 [==============================] - 8s 66ms/step - loss: 1.4213 - accuracy: 0.3524 - val_loss: 1.6843 - val_accuracy: 0.3599 - lr: 0.0010
Epoch 2/100
92/92 [==============================] - 6s 68ms/step - loss: 1.1945 - accuracy: 0.4357 - val_loss: 1.5478 - val_accuracy: 0.4959 - lr: 0.0010
Epoch 3/100
92/92 [==============================] - 6s 66ms/step - loss: 1.0669 - accuracy: 0.5163 - val_loss: 1.3805 - val_accuracy: 0.6238 - lr: 0.0010
Epoch 4/100
92/92 [==============================] - 6s 61ms/step - loss: 0.8713 - accuracy: 0.6333 - val_loss: 1.0978 - val_accuracy: 0.6878 - lr: 0.0010
Epoch 5/100
92/92 [==============================] - 6s 67ms/step - loss: 0.7308 - accuracy: 0.7049 - val_loss: 0.7364 - val_accuracy: 0.7646 - lr: 0.0010
Epoch 6/100
92/92 [==============================] - 6s 66ms/step - loss: 0.6616 - accuracy: 0.7333 - val_loss: 0.5617 - val_accuracy: 0.7980 - lr: 0.0010
Epoch 7/100
92/92 [==============================] - 6s 63ms/step - lo

: 